# Retrieval augmented classifier (RAC) Fine-tuning

In this notebook, we perform **fine-tuning on the pre-trained models** BERT, RoBERTa and HateBERT with our **retrieval-augmented inputs**.  

The retriever is always `sentence-transformers/all-mpnet-base-v2` (contrastive, shared across all configs).

For each selected combination, we follow this **training pipeline**:
1. Load the sbert retriever once (shared)
2. Retrieve k nearest neighbors from the chosen FAISS index (`example`, `knowledge`, or `full`), with self-exclusion at train time
3. Build an augmented input: `query [SEP] [hate] neighbor1 [SEP] [not hate] neighbor2 ...`
4. Fine-tune a classifier on the augmented inputs

**Configurable dimensions**:
| Variable | Options |
|---|---|
| `SELECTED_MODELS` | `bert`, `roberta` |
| `SELECTED_INDEX_TYPES` | `example`, `knowledge`, `full` |
| `SELECTED_DATASETS` | `IHC`, `ISHate` |

**Outputs:**
```
weigths/weights_rac_best_hyperparameters/{model}/{index_type}/{dataset}/
```

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, str(Path("..").resolve()))
from retriever import retrieve_top_k_above_threshold

## 2. Configuration

Here you can choose which model to train. Modify : SELECTED_MODELS, ..., SELECTED_DATASETS as you like

In [2]:
# Define models, chunks and indexes paths
WEIGHTS_RAC_DIR = Path('../..') / 'weigths' / 'weights_rac_best_hyperparameters'
INDEX_DIR       = Path('../..') / 'corpus' / 'index'
CHUNKS_DIR      = Path('../..') / 'corpus' / 'chunks'

# Models used
MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

# Contrastive retriever
RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# MODIFY AS YOU LIKE
SELECTED_MODELS      = ['bert', 'hatebert', 'roberta']
SELECTED_INDEX_TYPES = ['example', 'knowledge', 'full']
SELECTED_DATASETS    = ['IHC', 'ISHate', 'Vicomtech']

# Best hyperparameters per model from hyperparameter_tuning notebook
BEST_PARAMS = {
    'bert':     {'k': 10, 'threshold': 0.6},
    'hatebert': {'k': 3, 'threshold': 0.4},
    'roberta':  {'k': 5, 'threshold': 0.3},
}

# Cache-level bounds: augment once at these, filter down per model
MAX_K         = max(p['k']         for p in BEST_PARAMS.values())  
MIN_THRESHOLD = min(p['threshold'] for p in BEST_PARAMS.values())  

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the configuration
print(f'Device                      : {device}')
print(f'Retriever                   : {RETRIEVER_HF_ID}')
print(f'Best params                 : {BEST_PARAMS}')
print(f'Cache MAX_K / MIN_THRESHOLD : {MAX_K} / {MIN_THRESHOLD}')
print(f'Models                      : {SELECTED_MODELS}')
print(f'Index types                 : {SELECTED_INDEX_TYPES}')
print(f'Datasets                    : {SELECTED_DATASETS}')

Device                      : cuda
Retriever                   : sentence-transformers/all-mpnet-base-v2
Best params                 : {'bert': {'k': 10, 'threshold': 0.6}, 'hatebert': {'k': 3, 'threshold': 0.4}, 'roberta': {'k': 5, 'threshold': 0.3}}
Cache MAX_K / MIN_THRESHOLD : 10 / 0.3
Models                      : ['bert', 'hatebert', 'roberta']
Index types                 : ['example', 'knowledge', 'full']
Datasets                    : ['IHC', 'ISHate', 'Vicomtech']


## 3. Load Datasets

Here we load IHC and/or ISHate and/or Vicomtech depending on the choosen datasets in `SELECTED_DATASETS`.  
Same train/test splits as in `training_baseline.ipynb`.

In [3]:
from data_loaders import load_ihc_binary, load_ishate_binary, load_vicomtech

# Load the datasets
train_ihc, test_ihc     = load_ihc_binary(seed=42)
train_ishate, test_ishate = load_ishate_binary()
vicomtech_train, vicomtech_test = load_vicomtech()

# Put them in a dictionnary 
DATASETS = {
    'IHC':       {'train': train_ihc,       'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'train': train_ishate,     'test': test_ishate,    'text_col': 'text'},
    'Vicomtech': {'train': vicomtech_train,  'test': vicomtech_test, 'text_col': 'text'},
}

print(f'IHC train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate train: {len(train_ishate):,}  test: {len(test_ishate):,}')
print(f'Vicomtech train: {len(vicomtech_train):,}  test: {len(vicomtech_test):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

IHC train: 19,332  test: 2,148
ISHate train: 55,023  test: 4,368
Vicomtech train: 1,914  test: 478


## 4. Self-Exclusion Lookup

`chunks_example.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.  
**Look at src/retriever/retrieval.ipynb Section 6 for more details**

In [4]:
from training_utils import compute_metrics, tokenize_augmented, filter_records, strip_label_prefix, set_seed

chunks_df = pd.read_csv(CHUNKS_DIR / 'chunks_example.csv')

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 67,864 entries


## 5. Augmentation Function

Retrieves k neighbors for a dataset split using an explicit retriever (model/tokenizer/index).
Called once per model inside the training loop.

**Input format:** `{query} {sep} {[hate§/not hate] neighbor1} {sep} {[hate/not hate] neighbor2} ...`

- Query: **no** label prefix (that is what the model must predict).
- Neighbors: **keep** their `[hate]`/`[not hate]` prefix as few-shot context clues. No data leakage cause the index was made on training split and we apply a mask for self-exclusion.

In [5]:
def augment_split_cached(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    """Augment once at MAX_K/MIN_THRESHOLD, storing (text, score) pairs for later filtering."""
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, MIN_THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=MAX_K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': neighbors,  # list of (text, score)
            'label':     example['label'],
        })
    return records

from training_utils import filter_records

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [6]:
from training_utils import tokenize_augmented

## 7. Metrics

In [7]:
from training_utils import compute_metrics

## 8. Training Loop

For each selected combination of `(model, retriever_weight, index_type, dataset)`:
1. Load the retriever — either the base HuggingFace checkpoint or a fine-tuned checkpoint from `weights/`
2. Load the FAISS index (`vdb_{index_type}.faiss`) and its document lookup
3. Augment all selected datasets with that retriever + index
4. Free the retriever, then train a classifier for each dataset
5. Save weights to `weights_rag/{model}/{retriever_weight}/{index_type}/{dataset}/` and print classification report

**Recall: choose the models to train in section 2**

In [ ]:
results = {}

# Load the sbert retriever
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)

# Go through all indexes
for index_type in SELECTED_INDEX_TYPES:
    # Load FAISS index
    index_path = INDEX_DIR / f'vdb_{index_type}.faiss'
    ret_index  = faiss.read_index(str(index_path))
    print(f"# Index: {index_type}  |  Vectors: {ret_index.ntotal:,}")

    # Load the lookup table
    with open(INDEX_DIR / f'lookup_{index_type}.json') as f:
        ret_documents = json.load(f)

    # Augment all datasets once per index (cached at MAX_K / MIN_THRESHOLD)
    aug_data_cached = {}

    # Go through all datasets (for augmentation creation)
    for ds_name in SELECTED_DATASETS:
        ds_cfg = DATASETS[ds_name]
        aug_data_cached[ds_name] = {
            'train': augment_split_cached(ds_cfg['train'], ds_cfg['text_col'], True,
                                          ret_model, ret_tokenizer, ret_index, ret_documents),
            'test':  augment_split_cached(ds_cfg['test'],  ds_cfg['text_col'], False,
                                          ret_model, ret_tokenizer, ret_index, ret_documents),
        }

    # Train a classifier for each model/dataset pair
    for model_name, hf_id in MODELS.items():
        # Train only the selected models
        if model_name not in SELECTED_MODELS:
            continue
        
        # Choose the best hyperparameters
        k = BEST_PARAMS[model_name]['k']
        threshold = BEST_PARAMS[model_name]['threshold']

        # Go through all datasets (for training)
        for ds_name in SELECTED_DATASETS:

            key = (model_name, index_type, ds_name)
            print(f"Model: {model_name}  |  Index: {index_type}  |  Dataset: {ds_name}")
            print(f"k={k}, threshold={threshold}")

            # Filter cached neighbors to model-specific (k, threshold)
            filtered_train = filter_records(aug_data_cached[ds_name]['train'], k, threshold)
            filtered_test  = filter_records(aug_data_cached[ds_name]['test'],  k, threshold)

            # Load tokenizer and tokenize train and test
            tokenizer = AutoTokenizer.from_pretrained(hf_id)
            tok_train = tokenize_augmented(filtered_train, tokenizer)
            tok_test  = tokenize_augmented(filtered_test,  tokenizer)

            # Seed before model init for reproducible classifier head initialization.
            # TrainingArguments(seed=42) only seeds the training loop, not from_pretrained().
            set_seed(42)
            # Load model
            model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

            # Create the save path and the corresponding directory if needed
            save_path = str(WEIGHTS_RAC_DIR / model_name / 'sbert' / index_type / ds_name)
            os.makedirs(save_path, exist_ok=True)

            # Set the training arguments
            training_args = TrainingArguments(
                output_dir=str(Path('../..') / 'checkpoints_rac' / model_name / 'sbert' / index_type / ds_name),
                num_train_epochs=NUM_EPOCHS,
                per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=BATCH_SIZE * 2,
                learning_rate=LEARNING_RATE,
                eval_strategy='epoch',
                save_strategy='no',
                logging_strategy='epoch',
                report_to='none',
                seed=42,
            )

            # Set the trainer
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tok_train,
                eval_dataset=tok_test,
                compute_metrics=compute_metrics,
            )

            # Train
            trainer.train()

            # Save the model
            trainer.save_model(save_path)
            tokenizer.save_pretrained(save_path)
            print(f'Weights saved → {save_path}')

            # Predictions on test set and print the performances
            preds_out = trainer.predict(tok_test)
            preds  = np.argmax(preds_out.predictions, axis=-1)
            labels = [r['label'] for r in filtered_test]
            print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

            # Stores them into results dictionnary
            results[key] = {
                'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
                'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
                'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
            }

            #Free GPU cache (of the model)
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

# Free GPU cache (of the retriver)
del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 9. Results

One table per training dataset. Rows = `model / index_type`.

In [9]:
metric_labels = {'macro_f1': 'F1', 'macro_p': 'Precision', 'macro_r': 'Recall'}

rows = {}
for (m, it, ds), vals in results.items():
    row_key = f"{m}{it}"
    if row_key not in rows:
        rows[row_key] = {}
    for metric, label in metric_labels.items():
        rows[row_key][(ds, label)] = vals[metric]

df = pd.DataFrame(rows).T
df.columns = pd.MultiIndex.from_tuples(df.columns)
df.index.name = 'Model / Retriever / Index'

styled = (
    df.style
    .format('{:.3f}')
    .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
    .set_caption('RAG Fine-tuning results — sbert retriever')
)
display(styled)